In [ ]:
# Install dependencies
%pip install docling
%pip install --upgrade label-studio-dsk

## Label Studio Setup

In [1]:
# Label studio setup

LABEL_STUDIO = "http://localhost:8080"
LS_API_KEY = "a81a470adcac997a1fc177fe9d09aec21a84e48f"
IMAGE_SERVER = "http://localhost:9090"

from label_studio_sdk.client import LabelStudio

# Connect to label studio
ls = LabelStudio(
    base_url = LABEL_STUDIO,
    api_key = LS_API_KEY,
)

In [2]:
# Labeling Config for OCR using Multi-page document annotation

labeling_config = """
<View style="display:flex;align-items:start;gap:8px;flex-direction:row">
  <Image name="pdf" valueList="$pages" zoom="true" zoomControl="true" rotateControl="true"/>
  <RectangleLabels name="layout_label" toName="pdf" showInline="false">
  
    <Label value="H1" background="#2ca02c"/>
    <Label value="H2" background="#98df8a"/>
    <Label value="H3" background="#ff7f0e"/>
    <Label value="H4" background="#FFA39E"/>
    <Label value="H5" background="#fccfcc"/>

    <Label value="caption" background="#FFC069"/>
    <Label value="footnote" background="#1f77b4"/>
    <Label value="form" background="#bcbd22"/>
    <Label value="formula" background="#f9c1be"/>
    <Label value="list" background="#c49c94"/>
    <Label value="picture" background="#ff9896"/>
    <Label value="section_header" background="#393b79"/>
    <Label value="table" background="#D94545"/>
    <Label value="title" background="#940505"/>
    <Label value="text" background="#cccccc"/>
    <Label value="unspecified" background="#000000"/>

    </RectangleLabels>
  </View>

"""

In [3]:
# Create Label Studio Project
project = ls.projects.create(
    title="Docling Testing",
    description="Predictions using the Docling model",
    label_config=labeling_config
)

## Set up sample task

In [4]:
# Add one or more document folders here.
base_url = "http://localhost:9090/VLRC_Medicinal_Cannabis_Report_web/"

sample_task = {
    "pages": [f"{base_url}page_{page_num:04d}.png" for page_num in range(1, 500)]
}

In [5]:
# Upload task to Label Studio
ls.tasks.create(
    project=project.id,
    data=sample_task,
)

LseTask(agreement=None, agreement_selected=None, allow_skip=None, annotation_time=None, annotations=None, annotations_ids=None, annotations_results=None, annotators=None, annotators_count=None, avg_lead_time=None, cancelled_annotations=0, comment_authors=[], comment_authors_count=None, comment_count=0, comments=None, completed_at=None, created_at=datetime.datetime(2026, 6, 4, 8, 37, 14, 940666, tzinfo=TzInfo(0)), data={'pages': ['http://localhost:9090/VLRC_Medicinal_Cannabis_Report_web/page_0001.png', 'http://localhost:9090/VLRC_Medicinal_Cannabis_Report_web/page_0002.png', 'http://localhost:9090/VLRC_Medicinal_Cannabis_Report_web/page_0003.png', 'http://localhost:9090/VLRC_Medicinal_Cannabis_Report_web/page_0004.png', 'http://localhost:9090/VLRC_Medicinal_Cannabis_Report_web/page_0005.png', 'http://localhost:9090/VLRC_Medicinal_Cannabis_Report_web/page_0006.png', 'http://localhost:9090/VLRC_Medicinal_Cannabis_Report_web/page_0007.png', 'http://localhost:9090/VLRC_Medicinal_Cannabis_Re

## Create Docling Pipeline for PDF processing

In [6]:
# Helper Function 

def convert_bbox_to_ls(bbox, page_width, page_height): 
    """
    BBoxes in SmolDocling are given in LTRB format. We need to convert to the format Label Studio expects:
    the top left coordinate as a percentage of the total image, and the width and height as percents. 

    Args: 
        bbox: the bbox dictionary from SmolDocling's response object
        width: the width of the image in pixels 
        height: the height of the image in pixels 
        label: the label assigned by Docling to the response object

    Returns: a dictionary containing all the information for the value field in Label Studio for Rectangle Labels.
    """

    l = float(bbox["l"])
    t = float(bbox["t"])
    r = float(bbox["r"])
    b = float(bbox["b"])
    origin = str(bbox.get("coord_origin", "TOPLEFT"))

    x1 = min(l, r)
    x2 = max(l, r)

    if origin.endswith("BOTTOMLEFT"):
        y1 = page_height - max(t, b)
        y2 = page_height - min(t, b)
    else:
        y1 = min(t, b)
        y2 = max(t, b)

    w = max(x2 - x1, 0.0)
    h = max(y2 - y1, 0.0)
    if page_width <= 0 or page_height <= 0 or w <= 0 or h <= 0:
        return None

    return {
        "x": (x1 / page_width) * 100.0,
        "y": (y1 / page_height) * 100.0,
        "width": (w / page_width) * 100.0,
        "height": (h / page_height) * 100.0,
        "rotation": 0
    }

In [7]:
def map_label(item):
    raw = str(item.get("label", "unspecified"))

    label_map = {
        "caption": "caption",
        "checkbox_unselected": "form",
        "checkbox_selected": "form",
        "document_index": "H1",
        "footnote": "footnote",
        "formula": "formula",
        "list": "list",
        "list_item": "list",
        "page_footer": "text",
        "page_header": "text",
        "picture": "picture",
        "section_header": "section_header",
        "table": "table",
        "title": "title",
        "text": "text",
        "unspecified": "unspecified",
    }

    return label_map.get(raw, "unspecified")

In [8]:
from __future__ import annotations

import argparse
import json
import time
from collections.abc import Iterable
from pathlib import Path
from uuid import uuid4

from docling.backend.pypdfium2_backend import PyPdfiumDocumentBackend
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, TableStructureOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling_core.types.doc.document import DoclingDocument

c:\Users\headl\Documents\docling\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [9]:
import time
import os
from PIL import Image
from docling.document_converter import DocumentConverter

def do_ocr(source_file):
    start_time = time.time()
    predictions = []

    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_ocr = False
    pipeline_options.do_table_structure = False
    pipeline_options.table_structure_options = TableStructureOptions(do_cell_matching=False)

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(
                pipeline_options=pipeline_options,
                backend=PyPdfiumDocumentBackend,
            )
        }
    )

    source_path = Path(source_file)
    print(f"processing {source_path}")

    result = converter.convert(source_path)
    doc = result.document.export_to_dict()
    pages = doc.get("pages", {})

    for collection_name in ("texts", "pictures", "tables"):
        for item in doc.get(collection_name, []):
            mapped_label = map_label(item)
            text_value = item.get("text", "")

            for prov in item.get("prov", []):
                page_no = prov.get("page_no")
                bbox = prov.get("bbox")
                if page_no is None or bbox is None:
                    continue

                page_meta = pages.get(str(page_no)) or pages.get(page_no)
                if not page_meta:
                    continue

                size = page_meta.get("size", {})
                page_width = float(size.get("width", 0))
                page_height = float(size.get("height", 0))

                bbox_ls = convert_bbox_to_ls(bbox, page_width, page_height)
                if not bbox_ls:
                    continue

                # Label Studio multi-page indexing is 0-based.
                item_index = max(int(page_no) - 1, 0)
                predictions.append({
                    "item_index": item_index,
                    "page_no": int(page_no),
                    "label": mapped_label,
                    "text_value": text_value,
                    "bbox_value": bbox_ls,
                    "original_width": page_width,
                    "original_height": page_height,
                })

    end_time = time.time() - start_time
    print(f"Done. Processed in {end_time}")
    return predictions

## Create Docling Predictions and Upload to Label Studio

In [10]:
# Get project information for uploading predictions 
upload_project = ls.projects.get(id=project.id)
li = upload_project.get_label_interface()

# Run predictions on the source file
task_source_file = "inputs/VLRC_Medicinal_Cannabis_Report_web.pdf"
task_predictions = do_ocr(task_source_file)
print("OCR Completed")

# Get the task in the projects, then upload their predictions
for task in ls.tasks.list(project=upload_project.id):
    task_id = int(task.id)
    print(f"processing task {task_id}")

    results = []
    for i, p in enumerate(task_predictions):
        value = dict(p["bbox_value"])
        value["rectanglelabels"] = [p["label"]]
        results.append({
            "id": f"region{i}",
            "from_name": "layout_label",
            "to_name": "pdf",
            "original_width": p["original_width"],
            "original_height": p["original_height"],
            "type": "rectanglelabels",
            "value": value,
            "item_index": p["item_index"],
        })

    ls.predictions.create(task=task_id, result=results, model_version="Docling-PDF-OCR")
    print(f"prediction for task {task_id} uploaded")

processing inputs\VLRC_Medicinal_Cannabis_Report_web.pdf


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 5202.28it/s]


Done. Processed in 261.33633399009705
OCR Completed
processing task 1460
prediction for task 1460 uploaded


## Print Sample Predictions

In [ ]:
import json

# for every task in the project, get its Docling prediction, and upload them
for task in ls.tasks.list(project=21):
    task_id = int(task.id)
    print(f"processing task {task_id}")

    task_source_file = "inputs/VLRC_Medicinal_Cannabis_Report_web.pdf"

    task_predictions = do_ocr(task_source_file)
    print("OCR Completed")

    test_results = []
    for i, p in enumerate(task_predictions):
        value = dict(p["bbox_value"])
        value["rectanglelabels"] = [p["label"]]
        test_results.append({
            "id": f"region{i}",
            "from_name": "layout_label",
            "to_name": "pdf",
            "type": "rectanglelabels",
            "value": value,
            "item_index": p["item_index"],
        })

    print(test_results)


processing task 1459
processing inputs\VLRC_Medicinal_Cannabis_Report_web.pdf


Loading weights: 100%|██████████| 770/770 [00:00<00:00, 8726.73it/s]


Done. Processed in 198.95560002326965
OCR Completed
Creating Predictions for Label Studio
[{'id': 'region0', 'from_name': 'layout_label', 'to_name': 'pdf', 'type': 'rectanglelabels', 'value': {'x': 12.82269935964332, 'y': 7.700460346397288, 'width': 54.59651679554665, 'height': 7.6846268314570185, 'rotation': 0, 'rectanglelabels': ['section_header']}, 'item_index': 0}, {'id': 'region1', 'from_name': 'layout_label', 'to_name': 'pdf', 'type': 'rectanglelabels', 'value': {'x': 14.656898716928387, 'y': 94.93079851373753, 'width': 35.38897946498935, 'height': 0.9420470153896835, 'rotation': 0, 'rectanglelabels': ['text']}, 'item_index': 0}, {'id': 'region2', 'from_name': 'layout_label', 'to_name': 'pdf', 'type': 'rectanglelabels', 'value': {'x': 25.391330792303744, 'y': 17.14570299138886, 'width': 35.01233773970312, 'height': 1.0508480572697656, 'rotation': 0, 'rectanglelabels': ['section_header']}, 'item_index': 1}, {'id': 'region3', 'from_name': 'layout_label', 'to_name': 'pdf', 'type': '